<a href="https://colab.research.google.com/github/ML-HW-SYS/a3-AaronGoldblatt/blob/main/4-gemm_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMM on GPU

## 1. Set-up

In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [49]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a3-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a3-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull
%cd /content/gdrive/MyDrive/ece5545

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a3-{YOUR_HANDLE}"

mkdir: cannot create directory ‘/content/gdrive/MyDrive/ece5545’: File exists
/content/gdrive/MyDrive/ece5545
fatal: destination path 'a3-AaronGoldblatt' already exists and is not an empty directory.
/content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
Already on 'main'
Your branch is up to date with 'origin/main'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 518 bytes | 0 bytes/s, done.
From https://github.com/ML-HW-SYS/a3-AaronGoldblatt
   edd0461..6050b4c  main       -> origin/main
Updating edd0461..6050b4c
Fast-forward
 src/ops.py | 10 +++-------
 1 file changed, 3 insertions(+), 7 deletions(-)
/content/gdrive/MyDrive/ece5545


In [4]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

In [5]:
!ls {PROJECT_ROOT}

1-conv1d_cpu.ipynb  3-conv1d_fpga.ipynb  5-conv2d_dw_gpu.ipynb	README.md  tests
2-conv1d_gpu.ipynb  4-gemm_gpu.ipynb	 leaderboard_id.txt	src


## 2. Install TVM

In [6]:
!pip install numpy==1.24.3

In [7]:
!pip install tlcpack-nightly-cu102 -f https://tlcpack.ai/wheels

Looking in links: https://tlcpack.ai/wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.5/428.5 MB 3.8 MB/s eta 0:00:00


## 3. Check the implementation of `make_gemm_gpu_scheduler` function in `src.ops`

The function implements General Matrix Multiply (GEMM) on GPU. You should use TVM to optimize it.

Let $A \in \mathbb{R}^{m \times k}$, $W \in \mathbb{R}^{k \times n}$, and $B \in \mathbb{R}^{m \times n}$, then
$$
B = A \times W
$$
Please see the numpy matmul function for more detail: [link](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html).

The `make_gemm_gpu_scheduler` takes $m$, $k$, and $n$. The first matrix is $m \times k$, the second matrix is $k \times n$, and the output matrix is $m \times n$.

The function returns both the TVM scheduler and the TVM opterator for
1. Input $a$
2. Input $w$
3. Output $b$

The scheduler should be able to used to build a function with signature $func(a, w, b)$.
Please see the following cells for usage.

In [50]:
import tvm
import numpy as np
import sys
# Adding assignment 3 to the system path
# Make sure this matches your git directory
sys.path.insert(0, PROJECT_ROOT)
from src.ops import make_gemm_gpu_scheduler

M = 1024
N = 512
K = 2048
dtype = 'float32'
a_np = np.random.rand(M, K).astype(dtype)
w_np = np.random.rand(K, N).astype(dtype)
b_np = np.matmul(a_np, w_np)

s, A, W, B = make_gemm_gpu_scheduler(M, K, N)
func = tvm.build(s, [A, W, B], "cuda")

dev = tvm.cuda(0)
a = tvm.nd.array(a_np, dev)
w = tvm.nd.array(w_np, dev)
b = tvm.nd.array(np.zeros((M, N), dtype), dev)
func(a, w, b)
evaluator = func.time_evaluator(func.entry_name, dev, number=1, repeat =1)


print("Answer:", b_np)
print("Output:", b)
print(f"GEMM TVM: %f ms" % (evaluator(a, w, b).mean * 1e3))

Answer: [[515.5856  510.89746 494.21317 ... 506.01483 493.92487 496.78787]
 [524.4897  515.29785 508.3551  ... 515.07196 504.0684  505.93738]
 [525.7474  523.77545 504.60883 ... 520.58685 506.5599  512.12787]
 ...
 [504.58853 520.7596  501.0802  ... 514.2794  504.95444 507.8856 ]
 [525.82715 529.0434  508.78976 ... 519.55585 509.30658 513.1166 ]
 [523.83215 520.7783  512.1833  ... 519.5596  507.29224 508.89734]]
Output: [[515.5857  510.89798 494.21295 ... 506.01492 493.9248  496.78802]
 [524.49    515.29736 508.35483 ... 515.0716  504.06873 505.93707]
 [525.7473  523.77576 504.60907 ... 520.5871  506.5604  512.1273 ]
 ...
 [504.58896 520.7598  501.08063 ... 514.2791  504.95395 507.88583]
 [525.8275  529.0429  508.78976 ... 519.5554  509.30624 513.11694]
 [523.83234 520.778   512.1834  ... 519.56036 507.2926  508.89685]]
GEMM TVM: 6.360352 ms


In [52]:
print(tvm.lower(s, [A, W, B], simple_mode=True))

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(A: T.Buffer((1024, 2048), "float32"), B: T.Buffer((2048, 512), "float32"), C: T.Buffer((1024, 512), "float32")):
        T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
        blockIdx_y = T.launch_thread("blockIdx.y", 32)
        A_local = T.allocate([32], "float32", "local")
        B_local = T.allocate([1024], "float32", "local")
        blockIdx_x = T.env_thread("blockIdx.x")
        threadIdx_y = T.env_thread("threadIdx.y")
        threadIdx_x = T.env_thread("threadIdx.x")
        C_1 = T.Buffer((524288,), data=C.data)
        with T.launch_thread(blockIdx_x, 16):
            T.launch_thread(threadIdx_y, 32)
            T.launch_thread(threadIdx_x, 32)
            C_1[blockIdx_y * 16384 + threadIdx_y * 512 + blockIdx_x * 32 + threadIdx_x] = T.float32(0)
        for k_outer in range(64):
            A_local_1 = T.Buffer((3

In [51]:
%cd {PROJECT_ROOT}
!python -m pytest tests/test_gemm_gpu.py

/content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
======================================= test session starts ========================================
platform linux -- Python 3.11.11, pytest-8.3.5, pluggy-1.5.0
rootdir: /content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
plugins: anyio-4.9.0, typeguard-4.4.2, langsmith-0.3.23
collected 20 items                                                                                 

tests/test_gemm_gpu.py ....................                                                  [100%]

======================================= 20 passed in 21.13s ========================================
